In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import Image, display
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import vix_tcn_revin_xai_plus as vtrx
import posthoc_analysis_v2 as posthoc_mod

CSV_PATH = ROOT / "data" / "raw" / "timeseries_data.csv"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
bundle_path = ROOT / "outputs" / "bundles" / "best_tcn_bundle.pt"
if not bundle_path.exists():
    bundle_path = ROOT / "outputs" / "bundles" / "best_model_bundle.pt"

model, bundle_meta, snapshot = vtrx.load_model_bundle(str(bundle_path), device=DEVICE)
cfg = vtrx.Config(**snapshot["cfg"])
cfg.csv_path = str(CSV_PATH)

target_mode = snapshot.get("target_mode", bundle_meta.get("target_mode", "level"))
print("bundle_path :", bundle_path)
print("target_mode :", target_mode)

In [ ]:
df_raw = vtrx.load_frame(cfg.csv_path, cfg.index_col, list(cfg.drop_cols))

_, _, dl_te, meta = vtrx.build_dataloaders(
    df_raw=df_raw,
    target_col=cfg.target_col,
    seq_len=cfg.seq_len,
    batch_size=cfg.batch_size,
    train_ratio=cfg.train_ratio,
    val_ratio=cfg.val_ratio,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.pin_memory and DEVICE.type == "cuda"),
    persistent_workers=cfg.persistent_workers,
    target_mode=target_mode,
)

In [ ]:
RUN_POSTHOC = True

In [ ]:
if RUN_POSTHOC:
    posthoc_result = posthoc_mod.run_post_hoc_analysis_v2(
        model=model,
        meta=meta,
        dl_te=dl_te,
        cfg=cfg,
        target_col_raw=cfg.target_col,
        event_horizons=[5, 1],
        event_quantiles=[0.95, 0.98],
        enforce_nonoverlap=True,
        min_index_gap=cfg.seq_len,
        match_methods=["knn", "propensity"],
        n_neighbors=1,
        max_pairs=200,
        n_perm=2000,
        fdr_correction=True,
        random_deletion_trials=100,
        mask_percentile=90,
        last_k=5,
        save_dir=str(ROOT / "outputs" / "posthoc"),
        sanity_dir=str(ROOT / "outputs" / "sanity"),
        show=True,
        seed=42,
    )
else:
    raise RuntimeError("RUN_POSTHOC=False")

In [ ]:
posthoc_dir = ROOT / "outputs" / "posthoc"
sanity_dir = ROOT / "outputs" / "sanity"

print("=== posthoc files ===")
for f in sorted(posthoc_dir.glob("*")):
    print(f.name)

print("\n=== sanity files ===")
for f in sorted(sanity_dir.glob("*")):
    print(f.name)

In [ ]:
tests_df = pd.read_csv(posthoc_dir / "posthoc_tests.csv")
tests_df

In [ ]:
sig_df = tests_df[tests_df["p_fdr"] < 0.05].sort_values("p_fdr")
sig_df

In [ ]:
cam_df = pd.read_csv(posthoc_dir / "cam_summary_per_sample.csv")
cam_df.groupby("group")[["com_signed", "com_abs", "foc_signed", "foc_abs", "last_k_mass"]].mean()

In [ ]:
del_df = pd.read_csv(posthoc_dir / "deletion_scores.csv")
del_df.groupby("group")[["important_deletion_delta", "random_deletion_delta_mean"]].mean()

In [ ]:
for img_name in [
    "mean_cam_abs.png",
    "mean_cam_signed.png",
    "deletion_vs_random.png",
    "matching_balance_plot.png",
]:
    img_path = posthoc_dir / img_name
    if img_path.exists():
        display(Image(filename=str(img_path), width=900))

In [ ]:
shift_df = pd.read_csv(sanity_dir / "cam_shift_test.csv")
shift_df